In [1]:
# CARREGAR ARQUIVO PICKLE
# Carrega as princiapais variáveis já processadas, útil para economizar tempo
import pickle

caminho_arquivo = 'dados.pickle'
with open(caminho_arquivo, 'rb') as arquivo_entrada:
   transcricoes_dados_dict, transcricoes_lista_por_sentencas, transcricoes_lista_por_texto_completo,                 transcricoes_string_unica = pickle.load(arquivo_entrada)
del arquivo_entrada, caminho_arquivo


In [4]:
import spacy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import BisectingKMeans
import prince
from scipy.stats import chi2_contingency


def executar_chd_reinert(lista_uces, n_classes=4, classes_alvo={"NOUN", "ADJ", "VERB", "ADV"}):
    """
    Executa a aproximação do Método Reinert (CHD) sobre uma lista de Unidades de Contexto Elementar (UCEs).
    """
    print("1. Inicializando processamento linguístico...")
    nlp = spacy.load("pt_core_news_lg", disable=["parser", "ner"])

    uces_processadas = []

    for doc in nlp.pipe(lista_uces, batch_size=50):
        lema_uce = [
            token.lemma_.lower() for token in doc
            if token.pos_ in classes_alvo and token.is_alpha
        ]
        uces_processadas.append(" ".join(lema_uce))

    print("2. Construindo Matriz Lexical Binária...")
    vectorizer = CountVectorizer(binary=True, min_df=2)
    matriz_lexical = vectorizer.fit_transform(uces_processadas)
    vocabulario_inicial = vectorizer.get_feature_names_out()

    df_matriz_completa = pd.DataFrame(matriz_lexical.toarray(), columns=vocabulario_inicial)

    # SANITIZAÇÃO DE MATRIZ ESPARSA
    mascara_linhas_validas = df_matriz_completa.sum(axis=1) > 0
    df_matriz_valida = df_matriz_completa[mascara_linhas_validas].copy()

    mascara_colunas_validas = df_matriz_valida.sum(axis=0) > 0
    df_matriz_valida = df_matriz_valida.loc[:, mascara_colunas_validas].copy()

    n_linhas, n_colunas = df_matriz_valida.shape
    if n_colunas < 2 or n_linhas < 2:
        raise ValueError(f"Matriz insuficiente para AFC: {n_linhas}x{n_colunas}.")

    print("3. Executando Análise Fatorial de Correspondência (AFC)...")
    componentes = min(5, n_linhas - 1, n_colunas - 1)
    ca = prince.CA(n_components=componentes, n_iter=10, random_state=42)
    ca.fit(df_matriz_valida)
    coordenadas_afc = ca.row_coordinates(df_matriz_valida)

    print("4. Executando Agrupamento Divisivo (Bisecting K-Means)...")
    chd = BisectingKMeans(n_clusters=n_classes, random_state=42)
    labels_calculados = chd.fit_predict(coordenadas_afc.to_numpy())

    # REALINHAMENTO DE ÍNDICES
    labels_classes_completo = np.full(len(lista_uces), -1)
    labels_classes_completo[mascara_linhas_validas] = labels_calculados

    df_matriz_valida['Classe_CHD'] = labels_calculados

    print("5. Calculando Qui-quadrado para Vocabulário Característico...")
    # Extrai o vocabulário atualizado (após remoção de colunas zeradas)
    vocabulario_final = df_matriz_valida.columns.drop('Classe_CHD')
    resultados_vocabulario = extrair_vocabulario_caracteristico(df_matriz_valida, vocabulario_final, n_classes)

    return labels_classes_completo, resultados_vocabulario



def extrair_vocabulario_caracteristico(df_matriz, vocabulario, n_classes):
    """
    Calcula o Qui-quadrado de cada palavra em relação a cada classe gerada.
    """
    resultados = []

    for classe_atual in range(n_classes):
        # Separa a matriz: Classe atual vs Restante do Corpus
        matriz_classe = df_matriz[df_matriz['Classe_CHD'] == classe_atual].drop(columns=['Classe_CHD'])
        matriz_resto = df_matriz[df_matriz['Classe_CHD'] != classe_atual].drop(columns=['Classe_CHD'])

        n_uces_classe = len(matriz_classe)
        n_uces_resto = len(matriz_resto)

        if n_uces_classe == 0:
            continue

        for termo in vocabulario:
            # Ocorrências e Ausências na Classe
            oc_classe = matriz_classe[termo].sum()
            aus_classe = n_uces_classe - oc_classe

            # Ocorrências e Ausências no Resto
            oc_resto = matriz_resto[termo].sum()
            aus_resto = n_uces_resto - oc_resto

            # Matriz de Contingência
            tabela = np.array([
                [oc_classe, oc_resto],
                [aus_classe, aus_resto]
            ])

            # Cálculo do Qui-quadrado. Aplica-se a correção de Yates por padrão.
            try:
                chi2, p_valor, _, _ = chi2_contingency(tabela)
            except ValueError:
                chi2, p_valor = 0, 1

            # Filtra apenas termos significativos (p < 0.05) com associação positiva à classe
            if p_valor < 0.05 and (oc_classe / n_uces_classe) > (oc_resto / n_uces_resto):
                resultados.append({
                    'Classe': classe_atual,
                    'Termo': termo,
                    'Chi2': round(chi2, 2),
                    'P-Valor': p_valor,
                    'Freq_Classe': oc_classe
                })

    df_resultados = pd.DataFrame(resultados).sort_values(by=['Classe', 'Chi2'], ascending=[True, False])
    return df_resultados

# ==============================================================================
# EXEMPLO DE EXECUÇÃO
# ==============================================================================
if __name__ == "__main__":
    # Simulação de UCEs (Corpus de teste)
    corpus_teste = [
        "O deputado subiu ao plenário e discursou fortemente contra o projeto de lei de censura.",
        "A base do governo votou favoravelmente na comissão de ética ontem à noite.",
        "O parlamentar argumentou que a constituição garante a liberdade de expressão de todos.",
        "Na sessão legislativa, os deputados aprovaram a medida provisória com urgência.",
        "A economia do país precisa de reformas fiscais urgentes para gerar mais emprego.",
        "O mercado financeiro reagiu bem ao anúncio do ministro sobre a nova taxa de juros.",
        "A inflação e o desemprego afetam diretamente o poder de compra da população.",
        "Investimentos em infraestrutura são fundamentais para o crescimento econômico sustentável."
    ]

    # Executa a CHD definindo 2 classes (para este exemplo pequeno)
    classes, vocabulario_estatistico = executar_chd_reinert(corpus_teste, n_classes=2)

    print("\n--- RESULTADO DAS CLASSES POR UCE ---")
    for i, txt in enumerate(corpus_teste):
        print(f"Classe {classes[i]} | {txt}")

    print("\n--- VOCABULÁRIO CARACTERÍSTICO (Qui-Quadrado Significativo) ---")
    print(vocabulario_estatistico.to_string(index=False))

1. Inicializando processamento linguístico...
2. Construindo Matriz Lexical Binária...


ValueError: Matriz insuficiente para AFC: 2x1.

In [16]:
# Executa a CHD definindo 2 classes (para este exemplo pequeno)
classes, vocabulario_estatistico = executar_chd_reinert(transcricoes_lista_por_sentencas, n_classes=10)

1. Inicializando processamento linguístico...
2. Construindo Matriz Lexical Binária...
3. Executando Análise Fatorial de Correspondência (AFC)...
4. Executando Agrupamento Divisivo (Bisecting K-Means)...
5. Calculando Qui-quadrado para Vocabulário Característico...


In [17]:
amostra_analisada = transcricoes_lista_por_sentencas[0:1000]

print("\n--- RESULTADO DAS CLASSES POR UCE ---")
for i, txt in enumerate(amostra_analisada):
    # Limita a impressão no terminal para evitar sobrecarga visual
    if i >= 15:
        break
    print(f"Classe {classes[i]} | {txt}")


--- RESULTADO DAS CLASSES POR UCE ---
Classe 3 | Boa tarde, Brasil.
Classe 3 | Nós estamos aqui hoje por algo muito maior do que nós mesmos.
Classe 3 | Então, hoje eu quero que vocês prestem realmente bastante atenção no que eu vou falar.
Classe 3 | Desde a caminhada que nós fizemos.
Classe 3 | >> [aplausos] >>
Classe 3 | Desde o dia que nós fizemos a caminhada, eu vi crianças, eu vi pessoas idosas indo ali caminhar com a gente e com um coração realmente machucado, mas um coração que foi sendo restaurado ao longo dos quilômetros para poder lutar por esse país.
Classe 3 | E chegado ali no final, eu realmente quis realmente só dar um recado para todo mundo e orar, porque eu não tenho dúvidas de que tudo, e eu repito, tudo o que está acontecendo no nosso país, Deus não está com os olhos fechados pro nosso Brasil.
Classe 3 | Ele não está.
Classe 3 | Eu não tenho dúvidas disso.
Classe 3 | Então, por que que nós estamos aqui no fim das contas?
Classe 3 | E segura, porque hoje eu vou dar lap

In [18]:
import pandas as pd
import collections

# 1. Auditoria de distribuição global das classes
contagem_classes = collections.Counter(classes)
print("\n--- DISTRIBUIÇÃO GLOBAL DE CLASSES ---")
for classe, quantidade in sorted(contagem_classes.items()):
    if classe == -1:
        print(f"Classe {classe} (Descartadas por falta de léxico): {quantidade} UCEs")
    else:
        print(f"Classe {classe}: {quantidade} UCEs")

# 2. Amostragem estratificada (Exibe 2 exemplos de cada classe válida para verificação visual)
print("\n--- AMOSTRA ESTRATIFICADA POR CLASSE ---")
df_auditoria = pd.DataFrame({'Classe': classes, 'Texto': transcricoes_lista_por_sentencas[0:1000]})
df_validas = df_auditoria[df_auditoria['Classe'] != -1]

for classe in sorted(df_validas['Classe'].unique()):
    amostra = df_validas[df_validas['Classe'] == classe].head(2)
    for _, linha in amostra.iterrows():
        print(f"Classe {linha['Classe']} | {linha['Texto']}")



--- DISTRIBUIÇÃO GLOBAL DE CLASSES ---
Classe -1 (Descartadas por falta de léxico): 1080 UCEs
Classe 0: 2 UCEs
Classe 1: 2 UCEs
Classe 2: 4 UCEs
Classe 3: 16574 UCEs
Classe 4: 2 UCEs
Classe 5: 3 UCEs
Classe 6: 2 UCEs
Classe 7: 2 UCEs
Classe 8: 2 UCEs
Classe 9: 2 UCEs

--- AMOSTRA ESTRATIFICADA POR CLASSE ---


ValueError: All arrays must be of the same length